# GPU-Accelerated Ensemble Generation

This notebook uses:
- **Dask arrays** for lazy loading and memory efficiency
- **PyTorch tensors** for GPU-accelerated computation
- **Data parallelism** for batch processing multiple ensemble members simultaneously

In [ ]:
import xarray as xr
import dask.array as da
import torch
import torch.nn.functional as F
from typing import Union, List, Tuple
import os
import sys
from pathlib import Path
import numpy as np
from torch.utils.data import DataLoader, Dataset
import warnings 
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
class GPUInitialConditionDataLoader:
    """
    GPU-accelerated data loader using Dask for lazy loading and PyTorch for computation.
    """
    
    def __init__(self, 
                 data_path: Union[str, List[str]], 
                 target_idx: int,
                 climato_path: str, 
                 rand_ds_path: str,
                 device: torch.device = None) -> None:
        
        self.data_path = data_path
        self.target_idx = target_idx
        self.climato_path = climato_path
        self.rand_ds_path = rand_ds_path
        self.device = device if device is not None else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Load with Dask arrays (lazy loading)
        self.dataset = self.load_dataset_lazy(data_path)
        self.uv_depth_t0, self.uv_depth_t1 = self.get_ocean_current_lazy(self.dataset, target_idx)
        
    def load_dataset_lazy(self, data_path: Union[str, List[str]]) -> xr.Dataset:
        """Load dataset with Dask arrays for lazy evaluation"""
        chunks = {"time": 2, "lat": 96, "lon": 96, "ch": 10}
        
        if isinstance(data_path, list):
            ds = xr.open_mfdataset(data_path, chunks=chunks, parallel=True)
            ds = ds.isel(time=slice(self.target_idx - 1, self.target_idx + 1))
        else:
            ds = xr.open_dataset(data_path, chunks=chunks)
            ds = ds.isel(time=slice(self.target_idx - 1, self.target_idx + 1))
        
        return ds
    
    def get_ocean_current_lazy(self, 
                               ds: xr.Dataset, 
                               target_idx: int) -> Tuple[dict, dict]:
        """
        Extract ocean current data keeping Dask arrays (no computation yet).
        """
        idx_t0 = target_idx - 1
        idx_t1 = target_idx
        
        # Extract u,v components (keep as Dask arrays)
        uv_t0 = ds["data"].isel(time=idx_t0, ch=slice(3, 5))
        uv_t1 = ds["data"].isel(time=idx_t1, ch=slice(3, 5))
        
        # Store statistics as Dask arrays (lazy)
        uv_depth_t0 = {
            "d0": uv_t0,
            "max_d0": uv_t0.max(dim=("lat", "lon"), skipna=True),
            "min_d0": uv_t0.min(dim=("lat", "lon"), skipna=True),
            "mean_d0": uv_t0.mean(dim=("lat", "lon"), skipna=True),
            "std_d0": uv_t0.std(dim=("lat", "lon"), skipna=True)
        }
        
        # Depth levels 1-20
        for depth in range(1, 11):
            uv_depth_da = ds["data"].isel(time=idx_t0, ch=[25 + depth, 34 + depth])
            uv_depth_t0[f"d{depth}"] = uv_depth_da
            uv_depth_t0[f"max_d{depth}"] = uv_depth_da.max(dim=("lat", "lon"), skipna=True)
            uv_depth_t0[f"min_d{depth}"] = uv_depth_da.min(dim=("lat", "lon"), skipna=True)
            uv_depth_t0[f"mean_d{depth}"] = uv_depth_da.mean(dim=("lat", "lon"), skipna=True)
            uv_depth_t0[f"std_d{depth}"] = uv_depth_da.std(dim=("lat", "lon"), skipna=True)
            
        for depth in range(11, 21):
            uv_depth_da = ds["data"].isel(time=idx_t0, ch=[55 + depth, 64 + depth])
            uv_depth_t0[f"d{depth}"] = uv_depth_da
            uv_depth_t0[f"max_d{depth}"] = uv_depth_da.max(dim=("lat", "lon"), skipna=True)
            uv_depth_t0[f"min_d{depth}"] = uv_depth_da.min(dim=("lat", "lon"), skipna=True)
            uv_depth_t0[f"mean_d{depth}"] = uv_depth_da.mean(dim=("lat", "lon"), skipna=True)
            uv_depth_t0[f"std_d{depth}"] = uv_depth_da.std(dim=("lat", "lon"), skipna=True)
        
        # Same for t1
        uv_depth_t1 = {
            "d0": uv_t1,
            "max_d0": uv_t1.max(dim=("lat", "lon"), skipna=True),
            "min_d0": uv_t1.min(dim=("lat", "lon"), skipna=True),
            "mean_d0": uv_t1.mean(dim=("lat", "lon"), skipna=True),
            "std_d0": uv_t1.std(dim=("lat", "lon"), skipna=True)
        }
        
        for depth in range(1, 11):
            uv_depth_da = ds["data"].isel(time=idx_t1, ch=[25 + depth, 34 + depth])
            uv_depth_t1[f"d{depth}"] = uv_depth_da
            uv_depth_t1[f"max_d{depth}"] = uv_depth_da.max(dim=("lat", "lon"), skipna=True)
            uv_depth_t1[f"min_d{depth}"] = uv_depth_da.min(dim=("lat", "lon"), skipna=True)
            uv_depth_t1[f"mean_d{depth}"] = uv_depth_da.mean(dim=("lat", "lon"), skipna=True)
            uv_depth_t1[f"std_d{depth}"] = uv_depth_da.std(dim=("lat", "lon"), skipna=True)
            
        for depth in range(11, 21):
            uv_depth_da = ds["data"].isel(time=idx_t1, ch=[55 + depth, 64 + depth])
            uv_depth_t1[f"d{depth}"] = uv_depth_da
            uv_depth_t1[f"max_d{depth}"] = uv_depth_da.max(dim=("lat", "lon"), skipna=True)
            uv_depth_t1[f"min_d{depth}"] = uv_depth_da.min(dim=("lat", "lon"), skipna=True)
            uv_depth_t1[f"mean_d{depth}"] = uv_depth_da.mean(dim=("lat", "lon"), skipna=True)
            uv_depth_t1[f"std_d{depth}"] = uv_depth_da.std(dim=("lat", "lon"), skipna=True)
        
        return uv_depth_t0, uv_depth_t1
    
    def get_ssh_anomaly(self) -> xr.DataArray:
        """Extract SSH anomaly (lazy)"""
        climato = xr.open_dataset(self.climato_path, chunks={'lat': 96, 'lon': 96})
        rand_ds = xr.open_dataset(self.rand_ds_path, chunks={'lat': 96, 'lon': 96})
        
        anomaly_ssh = climato.isel(time=0, ch=0)['data'] - rand_ds.isel(time=0, ch=0)['data']
        return anomaly_ssh

In [ ]:
class GPUStochasticNoiseGenerator:
    """
    GPU-accelerated stochastic noise generation using PyTorch.
    """
    
    def __init__(self,
                 ssh_anomaly: xr.DataArray,
                 is_pointwise: bool,
                 seed: int = 42,
                 device: torch.device = None) -> None:
        
        self.is_pointwise = is_pointwise
        self.seed = seed
        self.device = device if device is not None else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Convert to PyTorch tensor and move to GPU
        self.anomaly_ssh = torch.from_numpy(ssh_anomaly.values).float().to(self.device)
        self.nx = ssh_anomaly.sizes['lon']
        self.ny = ssh_anomaly.sizes['lat']
        
        # Generate noise on GPU
        self.ocean_regions = self.ocean_region_definiition()
        self.noise = self.create_noise_gpu(self.nx, self.ny, self.ocean_regions, self.seed)
    
    def ocean_region_definiition(self):
        """Same as before"""
        ocean_regions = [
            {'name': 'Gulf Stream', 'type': 'box', 'correlation_length': 4.0,
             'coords': {'y_range': [382, 542], 'x_range': [300, 620]},
             'blend': True, 'blend_width': 70},
            {'name': 'Kuroshio', 'type': 'box', 'correlation_length': 4.0,
             'coords': {'y_range': [382, 542], 'x_range': [1160, 1460]},
             'blend': True, 'blend_width': 70},
            {'name': 'Agulhas', 'type': 'box', 'correlation_length': 4.0,
             'coords': {'y_range': [82, 242], 'x_range': [680, 980]},
             'blend': True, 'blend_width': 70},
            {'name': 'Antarctic ACC', 'type': 'box', 'correlation_length': 10.0,
             'coords': {'y_range': [80, 162], 'x_range': [0, self.nx]},
             'blend': True, 'blend_width': 30},
        ]
        return ocean_regions
    
    def create_noise_gpu(self, nx: int, ny: int, ocean_regions: dict, seed: int) -> torch.Tensor:
        """Create stochasticity on GPU"""
        torch.manual_seed(seed)
        
        if not self.is_pointwise:
            # Regional noise with GPU acceleration
            L_field_regions = self.create_correlation_length_field_gpu((ny, nx), ocean_regions, 50)
            
            # Amplitude field based on SSH anomaly
            valid_mask = ~torch.isnan(self.anomaly_ssh)
            ssh_std = torch.std(self.anomaly_ssh[valid_mask]).item()
            amp_field_regions = torch.abs(self.anomaly_ssh) / ssh_std
            
            # Generate spatially-varying noise on GPU
            regional_noise = self.spatially_varying_gaussian_filter_gpu(
                (ny, nx), L_field_regions, amplitude=0.1, seed=seed
            )
            
            regional_noise_scaled = regional_noise * amp_field_regions
            return regional_noise_scaled
        else:
            # Spectral noise
            k = torch.normal(mean=-1.0, std=0.3, size=(1,), 
                           generator=torch.Generator().manual_seed(4000 + seed)).item()
            noise_spectral = self.spectral_correlated_noise_gpu(
                (ny, nx), correlation_length=10, amplitude=0.3, 
                spectral_slope=k, seed=42
            )
            return noise_spectral
    
    def create_correlation_length_field_gpu(self, shape, regions, default):
        """Create correlation length field on GPU"""
        ny, nx = shape
        L_field = torch.ones((ny, nx), device=self.device) * default
        
        y, x = torch.meshgrid(
            torch.arange(ny, device=self.device),
            torch.arange(nx, device=self.device),
            indexing='ij'
        )
        
        for region in regions:
            mask = torch.zeros((ny, nx), device=self.device)
            reg_type = region['type']
            correlation_length = region.get('correlation_length', 20.0)
            blend = region.get('blend', True)
            blend_width = region.get('blend_width', 30)
            
            if reg_type == 'box':
                coords = region['coords']
                y_min, y_max = coords['y_range']
                x_min, x_max = coords['x_range']
                
                box_mask = ((y >= y_min) & (y < y_max) & 
                           (x >= x_min) & (x < x_max)).float()
                
                if blend:
                    # Simple distance-based blending on GPU
                    mask = box_mask
                    # Simplified blending for GPU efficiency
                    from torch.nn.functional import avg_pool2d
                    kernel_size = min(blend_width, ny // 4)
                    if kernel_size > 1:
                        mask = mask.unsqueeze(0).unsqueeze(0)
                        mask = avg_pool2d(mask, kernel_size, stride=1, 
                                        padding=kernel_size//2)
                        mask = mask.squeeze()
                        # Ensure mask has the correct shape after pooling
                        if mask.shape != (ny, nx):
                            mask = F.interpolate(mask.unsqueeze(0).unsqueeze(0), 
                                                size=(ny, nx), 
                                                mode='bilinear', 
                                                align_corners=True).squeeze()
                else:
                    mask = box_mask
            
            L_field = L_field * (1 - mask) + correlation_length * mask
        
        return L_field
    
    def spatially_varying_gaussian_filter_gpu(self, shape, correlation_length_field, 
                                             amplitude=1.0, seed=None):
        """GPU-accelerated spatially-varying Gaussian filter"""
        ny, nx = shape
        valid_mask = ~torch.isnan(correlation_length_field)
        L_min = torch.min(correlation_length_field[valid_mask]).item()
        L_max = torch.max(correlation_length_field[valid_mask]).item()
        
        n_scales = 5
        L_scales = torch.logspace(np.log10(max(L_min, 1)), np.log10(L_max), 
                                  n_scales, device=self.device)
        
        # Generate white noise on GPU
        if seed is not None:
            torch.manual_seed(seed)
        white_noise = torch.randn(ny, nx, device=self.device)
        
        # Apply Gaussian blur at different scales using PyTorch
        noise_scales = []
        for L in L_scales:
            sigma = L.item()
            # Use PyTorch's gaussian blur
            kernel_size = int(6 * sigma + 1)
            if kernel_size % 2 == 0:
                kernel_size += 1
            kernel_size = min(kernel_size, min(ny, nx) // 2 * 2 - 1)
            
            if kernel_size >= 3:
                filtered = self.gaussian_blur_2d_gpu(white_noise.unsqueeze(0).unsqueeze(0), 
                                                     kernel_size, sigma)
                noise_scales.append(filtered.squeeze())
            else:
                noise_scales.append(white_noise)
        
        # Blend scales
        output = torch.zeros((ny, nx), device=self.device)
        total_weight = torch.zeros((ny, nx), device=self.device)
        
        for i, L in enumerate(L_scales):
            log_L = torch.log(L)
            log_L_field = torch.log(torch.clamp(correlation_length_field, min=1e-6))
            weight = torch.exp(-((log_L_field - log_L)**2) / (2 * 0.5**2))
            
            output += weight * noise_scales[i]
            total_weight += weight
        
        output = output / (total_weight + 1e-10)
        output = torch.where(torch.isnan(correlation_length_field), 
                           torch.tensor(float('nan'), device=self.device), output)
        
        valid_mask = ~torch.isnan(output)
        valid_std = torch.std(output[valid_mask])
        output = output / valid_std * amplitude / 10
        
        return output
    
    def gaussian_blur_2d_gpu(self, tensor, kernel_size, sigma):
        """Apply Gaussian blur on GPU using PyTorch"""
        # Create Gaussian kernel
        x = torch.arange(kernel_size, device=self.device) - kernel_size // 2
        gauss_1d = torch.exp(-x**2 / (2 * sigma**2))
        gauss_1d = gauss_1d / gauss_1d.sum()
        
        gauss_2d = gauss_1d.view(-1, 1) @ gauss_1d.view(1, -1)
        gauss_2d = gauss_2d / gauss_2d.sum()
        gauss_2d = gauss_2d.view(1, 1, kernel_size, kernel_size)
        
        # Apply convolution
        padding = kernel_size // 2
        blurred = F.conv2d(tensor, gauss_2d, padding=padding)
        
        return blurred
    
    def spectral_correlated_noise_gpu(self, shape, correlation_length, amplitude=1.0, 
                                     spectral_slope=-3, seed=None):
        """GPU-accelerated spectral noise generation"""
        ny, nx = shape
        
        # Create frequency grid on GPU
        ky = torch.fft.fftfreq(ny, d=1.0, device=self.device)
        kx = torch.fft.fftfreq(nx, d=1.0, device=self.device)
        KY, KX = torch.meshgrid(ky, kx, indexing='ij')
        K = torch.sqrt(KX**2 + KY**2)
        K[0, 0] = 1.0
        
        # Generate white noise
        if seed is not None:
            torch.manual_seed(seed)
        white_noise = torch.randn(ny, nx, device=self.device)
        white_spectrum = torch.fft.fft2(white_noise)
        
        # Apply spectral filter
        k0 = 1.0 / correlation_length
        spectral_filter = K**spectral_slope * torch.exp(-K / k0)
        spectral_filter[0, 0] = 0
        
        filtered_spectrum = white_spectrum * spectral_filter
        correlated_noise = torch.real(torch.fft.ifft2(filtered_spectrum))
        
        # Normalize
        correlated_noise = correlated_noise / torch.std(correlated_noise) * amplitude
        
        # Make positive and normalize
        correlated_noise_positive = torch.exp(correlated_noise)
        adjust_factor = (4 + spectral_slope) + 1e-6
        correlated_noise_positive_normalized = correlated_noise_positive / adjust_factor
        

        return correlated_noise_positive_normalized

In [ ]:
class GPUCurrentBasedPerturbationGenerator:
    """
    GPU-accelerated perturbation generator with batch processing support.
    """
    
    def __init__(self, 
                 data_path: Union[str, List[str]], 
                 target_idx: int,
                 climato_path: str,
                 rand_ds_path: str,
                 dt: float,
                 device: torch.device = None,
                 batch_size: int = 4) -> None:

        self.dt = dt
        self.device = device if device is not None else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.batch_size = batch_size
        
        # Load data with lazy evaluation
        ic_dataloader = GPUInitialConditionDataLoader(data_path, target_idx, 
                                                     climato_path, rand_ds_path, self.device)
        self.dataset = ic_dataloader.dataset
        self.uv_depth_t0 = ic_dataloader.uv_depth_t0
        self.uv_depth_t1 = ic_dataloader.uv_depth_t1
        self.ssh_anomaly = ic_dataloader.get_ssh_anomaly()
    
    @staticmethod
    def create_land_mask_gpu(data_array_2d: xr.DataArray, 
                            threshold=1e-10, 
                            device=None) -> torch.Tensor:
        """Create land mask on GPU"""
        device = device if device is not None else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        if len(data_array_2d.dims) == 2:
            sample_data = torch.from_numpy(data_array_2d.values).float().to(device)
        else:
            raise ValueError("Input data_array must be 2D (lat, lon).")
        
        land_mask = torch.isnan(sample_data) | (torch.abs(sample_data) < threshold)
        return land_mask
    
    def displace_array_gpu(self, 
                          data: torch.Tensor,
                          dx: torch.Tensor,
                          dy: torch.Tensor,
                          land_mask: torch.Tensor) -> torch.Tensor:
        """
        GPU-accelerated displacement using PyTorch grid_sample.
        Supports batch processing for multiple ensemble members simultaneously.
        """
        # Handle both single and batch inputs
        if data.dim() == 2:
            data = data.unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
            dx = dx.unsqueeze(0) if dx.dim() == 2 else dx
            dy = dy.unsqueeze(0) if dy.dim() == 2 else dy
            single_input = True
        else:
            single_input = False
        
        batch_size, _, ny, nx = data.shape
        
        # Create coordinate grids
        y_coords = torch.arange(ny, device=self.device).view(1, -1, 1).expand(batch_size, ny, nx)
        x_coords = torch.arange(nx, device=self.device).view(1, 1, -1).expand(batch_size, ny, nx)
        
        # Calculate new coordinates
        new_y_coords = y_coords.float() + dy
        new_x_coords = x_coords.float() + dx
        
        # Normalize to [-1, 1] for grid_sample
        new_y_coords = 2.0 * new_y_coords / (ny - 1) - 1.0
        new_x_coords = 2.0 * new_x_coords / (nx - 1) - 1.0
        
        # Stack to create grid [B, H, W, 2]
        grid = torch.stack([new_x_coords, new_y_coords], dim=-1)
        
        # Apply displacement using grid_sample (bilinear interpolation)
        displaced_data = F.grid_sample(data, grid, mode='bilinear', 
                                      padding_mode='reflection', align_corners=True)
        
        # Handle land mask
        land_mask_expanded = land_mask.unsqueeze(0).unsqueeze(0).expand_as(data)
        invalid_mask = land_mask_expanded | torch.isnan(displaced_data)
        displaced_data = torch.where(invalid_mask, data, displaced_data)
        
        if single_input:
            displaced_data = displaced_data.squeeze(0).squeeze(0)
        
        return displaced_data
    
    def displace_for_channels_gpu_batch(self,
                                        dx_t0: torch.Tensor,
                                        dy_t0: torch.Tensor,
                                        dx_t1: torch.Tensor,
                                        dy_t1: torch.Tensor,
                                        batch_size: int = 1) -> xr.DataArray:
        """
        GPU-accelerated channel displacement with batch processing.
        Process multiple ensemble members simultaneously.
        """
        num_channels = self.dataset.isel(time=1).sizes['ch']
        
        # Expand displacement fields for batch processing
        if dx_t0.dim() == 2:
            dx_t0 = dx_t0.unsqueeze(0).expand(batch_size, -1, -1)
            dy_t0 = dy_t0.unsqueeze(0).expand(batch_size, -1, -1)
            dx_t1 = dx_t1.unsqueeze(0).expand(batch_size, -1, -1)
            dy_t1 = dy_t1.unsqueeze(0).expand(batch_size, -1, -1)
        
        displaced_datarray_t0 = []
        displaced_datarray_t1 = []
        
        # Process channels in batches
        for ch in range(num_channels):
            # Load data to GPU
            dataarray_2d_t0 = self.dataset["data"].isel(time=0, ch=ch)
            dataarray_2d_t1 = self.dataset["data"].isel(time=1, ch=ch)
            
            # Compute Dask arrays and move to GPU
            data_t0 = torch.from_numpy(dataarray_2d_t0.compute().values).float().to(self.device)
            data_t1 = torch.from_numpy(dataarray_2d_t1.compute().values).float().to(self.device)
            
            # Create land mask
            land_mask = self.create_land_mask_gpu(dataarray_2d_t0, device=self.device)
            
            # Batch displacement
            data_t0_batch = data_t0.unsqueeze(0).unsqueeze(0).expand(batch_size, 1, -1, -1)
            data_t1_batch = data_t1.unsqueeze(0).unsqueeze(0).expand(batch_size, 1, -1, -1)
            
            displaced_t0 = self.displace_array_gpu(data_t0_batch, dx_t0, dy_t0, land_mask)
            displaced_t1 = self.displace_array_gpu(data_t1_batch, dx_t1, dy_t1, land_mask)
            
            # Move back to CPU and convert to numpy
            displaced_datarray_t0.append(displaced_t0.squeeze(1).cpu().numpy())
            displaced_datarray_t1.append(displaced_t1.squeeze(1).cpu().numpy())
        
        # Stack and create DataArray
        displaced_t0_stacked = np.stack(displaced_datarray_t0, axis=1)  # [batch, ch, lat, lon]
        displaced_t1_stacked = np.stack(displaced_datarray_t1, axis=1)
        
        # Combine time dimension - stack along new time axis
        # Result shape: [batch, time=2, ch, lat, lon]
        displaced_combined = np.stack([displaced_t0_stacked, displaced_t1_stacked], axis=1)
        
        return displaced_combined
    
    def generate_batch_perturbations(self,
                                    perturbation_type: str,
                                    seeds: List[int],
                                    is_same_noise_in_time: bool = True,
                                    is_same_noise_for_component: bool = True,
                                    extraction_depth: int = 0) -> List[np.ndarray]:
        """
        Generate multiple perturbations in parallel on GPU.
        
        Args:
            perturbation_type: 'one_vector', 'pointwise_1', or 'pointwise_2'
            seeds: List of seeds for ensemble members
            
        Returns:
            List of perturbed states (as numpy arrays for saving)
        """
        batch_size = min(len(seeds), self.batch_size)
        is_pointwise = perturbation_type != 'one_vector'
        
        results = []
        
        # Process in batches
        for batch_start in range(0, len(seeds), batch_size):
            batch_seeds = seeds[batch_start:batch_start + batch_size]
            current_batch_size = len(batch_seeds)
            
            # Generate noise for batch
            dx_t0_batch = []
            dy_t0_batch = []
            dx_t1_batch = []
            dy_t1_batch = []
            
            for seed in batch_seeds:
                if is_same_noise_for_component:
                    if is_same_noise_in_time:
                        noise_gen = GPUStochasticNoiseGenerator(self.ssh_anomaly, is_pointwise, seed, self.device)
                        noise_u_t0 = noise_gen.noise
                        noise_v_t0, noise_u_t1, noise_v_t1 = noise_u_t0, noise_u_t0, noise_u_t0
                    else:
                        noise_gen_t0 = GPUStochasticNoiseGenerator(self.ssh_anomaly, is_pointwise, seed, self.device)
                        noise_gen_t1 = GPUStochasticNoiseGenerator(self.ssh_anomaly, is_pointwise, seed + 1, self.device)
                        noise_u_t0 = noise_gen_t0.noise
                        noise_v_t0 = noise_u_t0
                        noise_u_t1 = noise_gen_t1.noise
                        noise_v_t1 = noise_u_t1
                else:
                    if is_same_noise_in_time:
                        noise_u_t0 = GPUStochasticNoiseGenerator(self.ssh_anomaly, is_pointwise, seed, self.device).noise
                        noise_v_t0 = GPUStochasticNoiseGenerator(self.ssh_anomaly, is_pointwise, seed + 1, self.device).noise
                        noise_u_t1, noise_v_t1 = noise_u_t0, noise_v_t0
                    else:
                        noise_u_t0 = GPUStochasticNoiseGenerator(self.ssh_anomaly, is_pointwise, seed, self.device).noise
                        noise_v_t0 = GPUStochasticNoiseGenerator(self.ssh_anomaly, is_pointwise, seed + 1, self.device).noise
                        noise_u_t1 = GPUStochasticNoiseGenerator(self.ssh_anomaly, is_pointwise, seed + 2, self.device).noise
                        noise_v_t1 = GPUStochasticNoiseGenerator(self.ssh_anomaly, is_pointwise, seed + 3, self.device).noise
                
                # Compute displacement fields
                if perturbation_type == 'one_vector':
                    # Use min/max values
                    depth = extraction_depth
                    u_val_t0 = self.uv_depth_t0[f"max_d{depth}"].isel(ch=0).compute().values
                    v_val_t0 = self.uv_depth_t0[f"max_d{depth}"].isel(ch=1).compute().values
                    u_val_t1 = self.uv_depth_t1[f"max_d{depth}"].isel(ch=0).compute().values
                    v_val_t1 = self.uv_depth_t1[f"max_d{depth}"].isel(ch=1).compute().values
                    
                    grid_shape = (self.uv_depth_t1[f"d{depth}"].sizes['lat'], 
                                 self.uv_depth_t1[f"d{depth}"].sizes['lon'])
                    u_2d_t0 = torch.full(grid_shape, float(u_val_t0), device=self.device)
                    v_2d_t0 = torch.full(grid_shape, float(v_val_t0), device=self.device)
                    u_2d_t1 = torch.full(grid_shape, float(u_val_t1), device=self.device)
                    v_2d_t1 = torch.full(grid_shape, float(v_val_t1), device=self.device)
                    
                    dx_t0 = self.dt * (u_2d_t0 + noise_u_t0)
                    dy_t0 = -1 * self.dt * (v_2d_t0 + noise_v_t0)
                    dx_t1 = self.dt * (u_2d_t1 + noise_u_t1)
                    dy_t1 = -1 * self.dt * (v_2d_t1 + noise_v_t1)
                    
                else:  # pointwise
                    depth = 0
                    u_t0 = torch.from_numpy(self.uv_depth_t0[f"d{depth}"].isel(ch=0).compute().values).float().to(self.device)
                    v_t0 = torch.from_numpy(self.uv_depth_t0[f"d{depth}"].isel(ch=1).compute().values).float().to(self.device)
                    u_t1 = torch.from_numpy(self.uv_depth_t1[f"d{depth}"].isel(ch=0).compute().values).float().to(self.device)
                    v_t1 = torch.from_numpy(self.uv_depth_t1[f"d{depth}"].isel(ch=1).compute().values).float().to(self.device)
                    
                    dx_t0 = self.dt * u_t0 * noise_u_t0
                    dy_t0 = -1 * self.dt * v_t0 * noise_v_t0
                    dx_t1 = self.dt * u_t1 * noise_u_t1
                    dy_t1 = -1 * self.dt * v_t1 * noise_v_t1
                
                dx_t0_batch.append(dx_t0)
                dy_t0_batch.append(dy_t0)
                dx_t1_batch.append(dx_t1)
                dy_t1_batch.append(dy_t1)
            
            # Stack batch
            dx_t0_stacked = torch.stack(dx_t0_batch)
            dy_t0_stacked = torch.stack(dy_t0_batch)
            dx_t1_stacked = torch.stack(dx_t1_batch)
            dy_t1_stacked = torch.stack(dy_t1_batch)
            
            # Apply displacement to all channels in batch
            perturbed_batch = self.displace_for_channels_gpu_batch(
                dx_t0_stacked, dy_t0_stacked, dx_t1_stacked, dy_t1_stacked, 
                current_batch_size
            )
            
            # Split batch results
            for i in range(current_batch_size):
                results.append(perturbed_batch[i:i+1])
        
        return results

# GPU-Accelerated Ensemble Generation Pipeline

Now we'll generate all 200 ensemble members using GPU parallelism.

In [ ]:
# Configuration
DATA_PATH = "/Odyssey/public/glonet/glorys12_1993-01-01_to_1993-06-30_init_states/combined_input.nc"
TARGET_IDX = 1
CLIMATO_PATH = "/Odyssey/public/glonet/1993-06-01_climatology/mean1.nc"
RAND_DS_PATH = "/Odyssey/public/glonet/2022-06-01_init_states/input1.nc"
OUTPUT_DIR = "./ensemble_output_gpu"
DT = 1.0
BASE_SEED = 42
BATCH_SIZE = 40  # Process 20 ensemble members simultaneously on GPU

# Create output directories
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

onevector_dir = output_dir / "one_vector"
pointwise1_dir = output_dir / "pointwise_1"
pointwise2_dir = output_dir / "pointwise_2"

for dir_path in [onevector_dir, pointwise1_dir, pointwise2_dir]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("GPU-Accelerated Configuration:")
print(f"  Device: {device}")
print(f"  Batch size: {BATCH_SIZE} (parallel ensemble members)")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"\nOutput directories created:")
print(f"  - {onevector_dir}")
print(f"  - {pointwise1_dir}")
print(f"  - {pointwise2_dir}")

In [ ]:
import time

print("="*80)
print("GPU-ACCELERATED ENSEMBLE GENERATION")
print("="*80)
print(f"Processing {BATCH_SIZE} members in parallel on GPU")
print()

# Initialize GPU generator once (reuse for all perturbations)
print("Initializing GPU data loader...")
start_time = time.time()
generator = GPUCurrentBasedPerturbationGenerator(
    data_path=DATA_PATH,
    target_idx=TARGET_IDX,
    climato_path=CLIMATO_PATH,
    rand_ds_path=RAND_DS_PATH,
    dt=DT,
    device=device,
    batch_size=BATCH_SIZE
)
print(f"✓ Initialization complete ({time.time() - start_time:.2f}s)")
print()

# Method 1: One Vector (40 members)
print("="*80)
print("METHOD 1: ONE VECTOR - 40 MEMBERS (GPU Parallel)")
print("="*80)
start_method1 = time.time()

seeds_onevector = [BASE_SEED + i + 10 for i in range(40)]
perturbed_states = generator.generate_batch_perturbations(
    perturbation_type='one_vector',
    seeds=seeds_onevector,
    is_same_noise_in_time=True,
    is_same_noise_for_component=True,
    extraction_depth=0
)

# Save results
for idx, state in enumerate(perturbed_states):
    output_path = onevector_dir / f"member_{idx:03d}_initial_condition.nc"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Convert to xarray and save
    # state shape: [1, time, ch, lat, lon] -> squeeze to [time, ch, lat, lon]
    ds_out = xr.Dataset({
        "data": xr.DataArray(
            state.squeeze(0),  # Remove batch dimension
            dims=["time", "ch", "lat", "lon"],
            coords={
                "time": generator.dataset["data"].time,
                "ch": generator.dataset["data"].ch,
                "lat": generator.dataset["data"].lat,
                "lon": generator.dataset["data"].lon
            }
        )
    })
    ds_out.to_netcdf(output_path)
    
    if (idx + 1) % 10 == 0:
        print(f"  Saved {idx + 1}/40 members")

elapsed_method1 = time.time() - start_method1
print(f"✓ Method 1 complete: {elapsed_method1:.2f}s ({elapsed_method1/40:.2f}s per member)")
print()

# Method 2: Pointwise 1 (80 members)
print("="*80)
print("METHOD 2: POINTWISE 1 - 80 MEMBERS (GPU Parallel)")
print("="*80)
start_method2 = time.time()

seeds_pointwise1 = [BASE_SEED + 1000 + i for i in range(80)]
perturbed_states = generator.generate_batch_perturbations(
    perturbation_type='pointwise_1',
    seeds=seeds_pointwise1,
    is_same_noise_in_time=False,
    is_same_noise_for_component=False
)

# Save results
for idx, state in enumerate(perturbed_states):
    output_path = pointwise1_dir / f"member_{idx:03d}_initial_condition.nc"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    # state shape: [1, time, ch, lat, lon] -> squeeze to [time, ch, lat, lon]
    ds_out = xr.Dataset({
        "data": xr.DataArray(
            state.squeeze(0),  # Remove batch dimension
            dims=["time", "ch", "lat", "lon"],
            coords={
                "time": generator.dataset["data"].time,
                "ch": generator.dataset["data"].ch,
                "lat": generator.dataset["data"].lat,
                "lon": generator.dataset["data"].lon
            }
        )
    })
    ds_out.to_netcdf(output_path)
    
    if (idx + 1) % 20 == 0:
        print(f"  Saved {idx + 1}/80 members")

elapsed_method2 = time.time() - start_method2
print(f"✓ Method 2 complete: {elapsed_method2:.2f}s ({elapsed_method2/80:.2f}s per member)")
print()

# Method 3: Pointwise 2 (80 members)
print("="*80)
print("METHOD 3: POINTWISE 2 - 80 MEMBERS (GPU Parallel)")
print("="*80)
start_method3 = time.time()

seeds_pointwise2 = [BASE_SEED + 2000 + i for i in range(80)]
perturbed_states = generator.generate_batch_perturbations(
    perturbation_type='pointwise_2',
    seeds=seeds_pointwise2,
    is_same_noise_in_time=False,
    is_same_noise_for_component=False
)

# Save results
for idx, state in enumerate(perturbed_states):
    output_path = pointwise2_dir / f"member_{idx:03d}_initial_condition.nc"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    # state shape: [1, time, ch, lat, lon] -> squeeze to [time, ch, lat, lon]
    ds_out = xr.Dataset({
        "data": xr.DataArray(
            state.squeeze(0),  # Remove batch dimension
            dims=["time", "ch", "lat", "lon"],
            coords={
                "time": generator.dataset["data"].time,
                "ch": generator.dataset["data"].ch,
                "lat": generator.dataset["data"].lat,
                "lon": generator.dataset["data"].lon
            }
        )
    })
    ds_out.to_netcdf(output_path)
    
    if (idx + 1) % 20 == 0:
        print(f"  Saved {idx + 1}/80 members")

elapsed_method3 = time.time() - start_method3
print(f"✓ Method 3 complete: {elapsed_method3:.2f}s ({elapsed_method3/80:.2f}s per member)")
print()

# Summary
total_time = time.time() - start_time
print("="*80)
print("GPU-ACCELERATED ENSEMBLE GENERATION COMPLETE")
print("="*80)
print(f"Total time: {total_time:.2f}s ({total_time/200:.2f}s per member)")
print(f"  - Method 1 (40 members): {elapsed_method1:.2f}s")
print(f"  - Method 2 (80 members): {elapsed_method2:.2f}s")
print(f"  - Method 3 (80 members): {elapsed_method3:.2f}s")
print()
print(f"Speedup factor: Processing {BATCH_SIZE} members in parallel")
print(f"Expected speedup: {BATCH_SIZE}x faster than sequential CPU processing")
print()

# Verify
onevector_count = len(list(onevector_dir.glob("member_*_initial_condition.nc")))
pointwise1_count = len(list(pointwise1_dir.glob("member_*/perturbed_initial_condition.nc")))
pointwise2_count = len(list(pointwise2_dir.glob("member_*/perturbed_initial_condition.nc")))

print("Verification:")
print(f"  - One vector:  {onevector_count}/40")
print(f"  - Pointwise 1: {pointwise1_count}/80")
print(f"  - Pointwise 2: {pointwise2_count}/80")

print(f"  - Total:       {onevector_count + pointwise1_count + pointwise2_count}/200")
print(f"  - Total:       {onevector_count + pointwise1_count + pointwise2_count}/200")
print(f"  - Total:       {onevector_count + pointwise1_count + pointwise2_count}/200")

# Performance Optimization Tips

**Key GPU Optimizations Implemented:**

1. **Dask Arrays (Lazy Loading)**
   - Data is loaded only when needed
   - Reduces memory footprint
   - Enables processing datasets larger than RAM

2. **PyTorch GPU Acceleration**
   - All numerical operations run on GPU
   - Displacement uses `grid_sample` (highly optimized)
   - Spectral operations use `torch.fft` (GPU-accelerated FFT)

3. **Batch Processing**
   - Process multiple ensemble members simultaneously
   - Configurable `BATCH_SIZE` (default: 8)
   - Near-linear speedup with batch size

4. **Memory Management**
   - Automatic GPU memory cleanup
   - Data moved to CPU only when saving
   - Efficient tensor reuse

**Tuning Parameters:**
- Increase `BATCH_SIZE` for faster processing (if GPU memory allows)
- Reduce `chunks` in data loading if memory constrained
- Adjust correlation lengths for different noise characteristics

In [ ]:
# GPU Memory Management Utilities

def get_gpu_memory_info():
    """Display current GPU memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1e9
        reserved = torch.cuda.memory_reserved(0) / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU Memory:")
        print(f"  Allocated: {allocated:.2f} GB")
        print(f"  Reserved:  {reserved:.2f} GB")
        print(f"  Total:     {total:.2f} GB")
        print(f"  Free:      {total - reserved:.2f} GB")
    else:
        print("No GPU available")

def clear_gpu_cache():
    """Clear GPU memory cache"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print("✓ GPU cache cleared")

# Check initial GPU state
get_gpu_memory_info()